# BilateralFusionNet — ODIR-5K Multi-Label Classification
### With full checkpoint resume support
If training stops for any reason, just re-run from Cell 1 — it will automatically pick up from the last completed epoch.

In [1]:
# Cell 1: Install
import subprocess
subprocess.run(['pip', 'install', 'shap', 'grad-cam', 'timm', '-q'])
print('Done.')

In [2]:
# Cell 2: Imports
import os, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    multilabel_confusion_matrix, roc_curve, auc
)
from sklearn.model_selection import train_test_split

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import shap

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ⚡ SPEED OPTIMIZATIONS — Applied globally
torch.backends.cudnn.benchmark = True      # Auto-tune conv kernels (3-5% speedup)
torch.backends.cuda.matmul.allow_tf32 = True  # TF32 matmul (~2x speedup)
torch.backends.cudnn.allow_tf32 = True     # TF32 convolutions (~2x speedup)
print('✅ CuDNN benchmark + TF32 enabled.')


In [3]:
# Cell 2B: GPU Verification
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU name       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - GO TO SETTINGS'}")
print(f"GPU memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

if not torch.cuda.is_available():
    raise RuntimeError("GPU NOT ACTIVE. Go to Kaggle Settings → Accelerator → GPU T4 x2 → Save, then restart kernel.")

In [4]:
# Cell 3: Configuration
DATA_ROOT = Path('/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k')

# ✅ Correct image directory
TRAIN_IMG_DIR = DATA_ROOT / 'ODIR-5K' / 'ODIR-5K' / 'Training Images'

# ✅ Correct annotation file
ANNO_CSV = DATA_ROOT / 'full_df.csv'

print(f'Dataset root: {DATA_ROOT}')
print(f'Image directory: {TRAIN_IMG_DIR}')
print(f'CSV file: {ANNO_CSV}')

# Checkpoint paths
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
BEST_CKPT      = f'{CHECKPOINT_DIR}/best_model.pth'
LATEST_CKPT    = f'{CHECKPOINT_DIR}/latest_checkpoint.pth'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 64    # ⚡ Doubled: 2x throughput on T4 GPU
NUM_EPOCHS   = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 8    # ⚡ More parallel data loading workers
PREFETCH_FACTOR = 4  # ⚡ Prefetch batches ahead of GPU
PATIENCE     = 7

CLASSES    = ['Normal (N)', 'Diabetic Retinopathy (D)', 'Glaucoma (G)',
              'Cataract (C)', 'AMD (A)', 'Hypertension (H)',
              'Pathological Myopia (M)', 'Other (O)']
CLASS_ABBR = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
NUM_CLASSES = len(CLASSES)

print(f'Checkpoint dir   : {CHECKPOINT_DIR}')
print(f'Best checkpoint  : {BEST_CKPT}')
print(f'Latest checkpoint: {LATEST_CKPT}')

In [5]:
# Cell 4: Data Loading
def load_odir_annotations(anno_path, img_dir):
    anno_path = Path(anno_path)
    df = pd.read_csv(anno_path) if str(anno_path).endswith('.csv') else pd.read_excel(anno_path)
    print('Columns:', df.columns.tolist())
    print('Shape:',   df.shape)
    return df

def standardize_df(df, img_dir):
    img_dir = Path(img_dir)
    rename_map = {}
    col_lower  = {c.lower().strip(): c for c in df.columns}

    for k in ['id', 'patient id', 'patient_id']:
        if k in col_lower: rename_map[col_lower[k]] = 'id'
    for k in ['age', 'patient age']:
        if k in col_lower: rename_map[col_lower[k]] = 'age'
    for k in ['gender', 'patient sex', 'sex']:
        if k in col_lower: rename_map[col_lower[k]] = 'gender'
    for k in ['left-fundus', 'left fundus', 'left_fundus', 'filename_left']:
        if k in col_lower: rename_map[col_lower[k]] = 'left_img'
    for k in ['right-fundus', 'right fundus', 'right_fundus', 'filename_right']:
        if k in col_lower: rename_map[col_lower[k]] = 'right_img'
    for abbr in CLASS_ABBR:
        for k in [abbr.lower(), abbr.upper(), f'label_{abbr}']:
            if k in col_lower: rename_map[col_lower[k]] = abbr

    df = df.rename(columns=rename_map)

    def find_image(fname):
        if pd.isna(fname): return None
        p = img_dir / str(fname)
        if p.exists(): return str(p)
        for ext in ['.jpg', '.jpeg', '.png']:
            p2 = img_dir / (str(fname).split('.')[0] + ext)
            if p2.exists(): return str(p2)
        return None

    df['left_path']  = df['left_img'].apply(find_image)
    df['right_path'] = df['right_img'].apply(find_image)
    before = len(df)
    df = df.dropna(subset=['left_path', 'right_path'])
    print(f'Dropped {before-len(df)} rows with missing images. Remaining: {len(df)}')

    df['gender_enc'] = df['gender'].apply(
        lambda x: 1 if str(x).lower() in ['male','m','1'] else 0
    ) if 'gender' in df.columns else 0
    df['age'] = pd.to_numeric(df.get('age', 50), errors='coerce').fillna(50)
    df['age_norm'] = df['age'] / 100.0

    for abbr in CLASS_ABBR:
        if abbr not in df.columns: df[abbr] = 0
        df[abbr] = pd.to_numeric(df[abbr], errors='coerce').fillna(0).astype(int)
    return df

raw_df = load_odir_annotations(ANNO_CSV, TRAIN_IMG_DIR)
df     = standardize_df(raw_df, TRAIN_IMG_DIR)
print(f'\nTotal patients: {len(df)}')
for abbr, name in zip(CLASS_ABBR, CLASSES):
    c = df[abbr].sum()
    print(f'  {abbr} ({name[:25]:25s}): {c:4d} ({100*c/len(df):.1f}%)')

In [6]:
# Cell 5: EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('ODIR-5K Dataset EDA', fontsize=16, fontweight='bold')

counts = [df[a].sum() for a in CLASS_ABBR]
bars = axes[0,0].bar(CLASS_ABBR, counts, color='steelblue')
axes[0,0].set_title('Label Distribution')
for b, c in zip(bars, counts):
    axes[0,0].text(b.get_x()+b.get_width()/2, b.get_height()+5, str(c), ha='center', fontsize=9)

axes[0,1].hist(df['age'].dropna(), bins=30, color='coral')
axes[0,1].set_title('Age Distribution')
axes[0,1].axvline(df['age'].mean(), color='darkred', linestyle='--',
                  label=f'Mean: {df["age"].mean():.1f}')
axes[0,1].legend()

gc = df['gender'].value_counts()
axes[0,2].pie(gc.values, labels=gc.index, autopct='%1.1f%%',
              colors=['steelblue','coral'])
axes[0,2].set_title('Gender Distribution')

cooccur = df[CLASS_ABBR].values.T @ df[CLASS_ABBR].values
sns.heatmap(cooccur, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_ABBR, yticklabels=CLASS_ABBR, ax=axes[1,0])
axes[1,0].set_title('Co-occurrence Matrix')

lpp = df[CLASS_ABBR].sum(axis=1)
axes[1,1].hist(lpp, bins=range(0,6), color='mediumseagreen', align='left', rwidth=0.8)
axes[1,1].set_title('Labels per Patient')
axes[1,1].set_xticks(range(0,6))
multi = (lpp>1).mean()*100
axes[1,1].set_xlabel(f'{multi:.1f}% have >1 disease — confirms multi-label need')

try:
    s = df.iloc[0]
    li = np.array(Image.open(s['left_path']).resize((112,112)))
    ri = np.array(Image.open(s['right_path']).resize((112,112)))
    axes[1,2].imshow(np.concatenate([li,ri],axis=1))
    axes[1,2].set_title(f'Sample pair | Labels: {[a for a in CLASS_ABBR if s[a]==1]}')
    axes[1,2].axis('off')
except: axes[1,2].text(0.5,0.5,'Preview unavailable',ha='center',va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [8]:
# Cell 6: Dataset Class
class ODIRDataset(Dataset):
    def __init__(self, df, transform=None, augment=False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.cache     = {}                          # ← ADDED
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.1, 0.05),
        ]) if augment else None

    def __len__(self): return len(self.df)

    def load_image(self, path):                      # ← ENTIRE METHOD CHANGED
        if path in self.cache:
            return self.cache[path].copy()
        try:
            img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)  # ⚡ Pre-resize at load
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 0)
        if len(self.cache) < 10000:  # ⚡ Larger cache = fewer disk reads
            self.cache[path] = img.copy()
        return img

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        left  = self.load_image(row['left_path'])
        right = self.load_image(row['right_path'])
        if self.aug:
            left  = self.aug(left)
            right = self.aug(right)
        if self.transform:
            left  = self.transform(left)
            right = self.transform(right)
        meta   = torch.tensor([float(row.get('age_norm',0.5)),
                                float(row.get('gender_enc',0))], dtype=torch.float32)
        labels = torch.tensor([int(row[a]) for a in CLASS_ABBR], dtype=torch.float32)
        return left, right, meta, labels

base_transform = transforms.Compose([
    # ⚡ Resize removed: images are pre-resized at load time in load_image()
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
print('Dataset class ready.')

In [9]:
# Cell 7: Split and DataLoaders
df['label_str'] = df[CLASS_ABBR].apply(
    lambda r: ''.join([a for a in CLASS_ABBR if r[a]==1]) or 'N', axis=1)

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42,
    stratify=df['label_str'].apply(lambda x: x[0]))
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42,
    stratify=temp_df['label_str'].apply(lambda x: x[0]))

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

train_dataset = ODIRDataset(train_df, base_transform, augment=True)
val_dataset   = ODIRDataset(val_df,   base_transform, augment=False)
test_dataset  = ODIRDataset(test_df,  base_transform, augment=False)

train_loader = DataLoader(
    train_dataset, BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True,        # ⚡ Reuse workers across epochs
    prefetch_factor=PREFETCH_FACTOR, # ⚡ Prefetch 4 batches ahead
    drop_last=True,                  # ⚡ Avoids slow partial last batch
    multiprocessing_context='fork',  # ⚡ Faster worker start on Linux/Kaggle
)
val_loader   = DataLoader(
    val_dataset, BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR,
    multiprocessing_context='fork',
)
test_loader  = DataLoader(
    test_dataset, BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR,
    multiprocessing_context='fork',
)

# Sanity check
l, r, m, lb = next(iter(train_loader))
print(f'Batch shapes → left:{l.shape} right:{r.shape} meta:{m.shape} labels:{lb.shape}')

In [10]:
# Cell 8: Model Architecture
class CrossAttentionFusion(nn.Module):
    def __init__(self, feat_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn_L2R   = nn.MultiheadAttention(feat_dim, num_heads, dropout, batch_first=True)
        self.attn_R2L   = nn.MultiheadAttention(feat_dim, num_heads, dropout, batch_first=True)
        self.norm_L      = nn.LayerNorm(feat_dim)
        self.norm_R      = nn.LayerNorm(feat_dim)
        self.ffn_L       = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(feat_dim*2, feat_dim),
                                         nn.Dropout(dropout))
        self.ffn_R       = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(feat_dim*2, feat_dim),
                                         nn.Dropout(dropout))
        self.norm_ffn_L  = nn.LayerNorm(feat_dim)
        self.norm_ffn_R  = nn.LayerNorm(feat_dim)

    def forward(self, f_left, f_right):
        qL = f_left.unsqueeze(1);  qR = f_right.unsqueeze(1)
        aL, _ = self.attn_L2R(qL, qR, qR);  aL = aL.squeeze(1)
        fL = self.norm_L(f_left + aL)
        fL = self.norm_ffn_L(fL + self.ffn_L(fL))
        aR, _ = self.attn_R2L(qR, qL, qL);  aR = aR.squeeze(1)
        fR = self.norm_R(f_right + aR)
        fR = self.norm_ffn_R(fR + self.ffn_R(fR))
        return torch.cat([fL, fR], dim=1)

class MetadataMLP(nn.Module):
    def __init__(self, in_dim=2, hid=64, out=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid), nn.BatchNorm1d(hid), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(hid, out),   nn.BatchNorm1d(out),  nn.ReLU(True))
    def forward(self, x): return self.net(x)

class BilateralFusionNet(nn.Module):
    def __init__(self, num_classes=8, dropout=0.3):
        super().__init__()
        self.encoder = timm.create_model('efficientnet_b3', pretrained=True,
                                         num_classes=0, global_pool='avg')
        fd = self.encoder.num_features
        nh = 8 if fd % 8 == 0 else 4
        self.cross_attn  = CrossAttentionFusion(fd, nh, 0.1)
        self.meta_mlp    = MetadataMLP(2, 64, 128, 0.2)
        fusion_dim       = fd * 2 + 128
        self.classifier  = nn.Sequential(
            nn.Linear(fusion_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(512, 256),        nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(dropout/2),
            nn.Linear(256, num_classes))
        self.feat_dim    = fd
        self.num_classes = num_classes

    def forward(self, left, right, meta):
        fL = self.encoder(left)
        fR = self.encoder(right)
        fb = self.cross_attn(fL, fR)
        fm = self.meta_mlp(meta)
        return self.classifier(torch.cat([fb, fm], dim=1))

model = BilateralFusionNet(NUM_CLASSES).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'BilateralFusionNet | Parameters: {total:,}')
with torch.no_grad():
    o = model(torch.randn(2,3,224,224).to(DEVICE),
               torch.randn(2,3,224,224).to(DEVICE),
               torch.randn(2,2).to(DEVICE))
    print(f'Forward pass output shape: {o.shape}  (expected [2, 8])')

# ⚡ torch.compile — JIT-compiles the model graph (PyTorch 2.x)
# Provides 10-30% speedup by fusing operations and reducing overhead
import torch._dynamo
torch._dynamo.config.suppress_errors = True  # Graceful fallback if unsupported op
try:
    model_compiled = torch.compile(model, mode='reduce-overhead')
    # Quick warmup to trigger compilation
    with torch.no_grad(), torch.cuda.amp.autocast():
        _ = model_compiled(torch.randn(2,3,224,224).to(DEVICE),
                            torch.randn(2,3,224,224).to(DEVICE),
                            torch.randn(2,2).to(DEVICE))
    model = model_compiled
    print('✅ torch.compile() applied — graph-level optimization active.')
except Exception as e:
    print(f'⚠️  torch.compile() skipped ({e}) — using eager model.')

In [11]:
# Cell 9: Focal Loss
class MultiLabelFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.pos_weight = pos_weight
    def forward(self, logits, targets):
        probs  = torch.sigmoid(logits)
        bce    = F.binary_cross_entropy_with_logits(
                     logits, targets, pos_weight=self.pos_weight, reduction='none')
        p_t    = probs * targets + (1 - probs) * (1 - targets)
        focal  = (1 - p_t) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * focal * bce).mean()

lc   = np.array([train_df[a].sum() for a in CLASS_ABBR])
pw   = np.clip((len(train_df) - lc) / (lc + 1e-6), 1.0, 10.0)
pos_weight_tensor = torch.tensor(pw, dtype=torch.float32).to(DEVICE)
criterion = MultiLabelFocalLoss(0.25, 2.0, pos_weight_tensor)
print('Focal loss ready. Positive weights:', {a: round(w,2) for a,w in zip(CLASS_ABBR,pw)})

In [12]:
# ============================================================
# Cell 10: Optimizer, Scheduler, and CHECKPOINT RESUME
# ============================================================
#
# HOW RESUME WORKS:
#   - Every epoch saves a 'latest_checkpoint.pth' with FULL state
#   - If that file exists when you re-run this cell, training
#     automatically continues from the next epoch
#   - If it does NOT exist, training starts from epoch 1
#   - Best checkpoint is saved separately and never overwritten
#     unless a better epoch is found
# ============================================================

encoder_params   = list(model.encoder.parameters())
new_layer_params = (list(model.cross_attn.parameters()) +
                    list(model.meta_mlp.parameters())   +
                    list(model.classifier.parameters()))

optimizer = optim.AdamW(
    [{'params': encoder_params,   'lr': LR * 0.1},
     {'params': new_layer_params, 'lr': LR}],
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)
scaler = torch.cuda.amp.GradScaler(init_scale=2048) if DEVICE.type == 'cuda' else None  # ⚡ Better init scale for FP16

# ---- Initialize training state ----
START_EPOCH    = 1
best_val_auc   = 0.0
patience_count = 0
history = {'train_loss':[], 'val_loss':[],
           'train_auc':[],  'val_auc':[],
           'train_f1':[],   'val_f1':[]}

# ---- Try to resume ----
if os.path.exists(LATEST_CKPT):
    print('=' * 55)
    print(f'CHECKPOINT FOUND: {LATEST_CKPT}')
    print('Restoring full training state...')
    print('=' * 55)

    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE)

    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if scaler is not None and ckpt.get('scaler_state_dict'):
        scaler.load_state_dict(ckpt['scaler_state_dict'])

    START_EPOCH    = ckpt['epoch'] + 1
    best_val_auc   = ckpt['best_val_auc']
    patience_count = ckpt['patience_count']
    history        = ckpt['history']

    print(f'  Last completed epoch : {ckpt["epoch"]}')
    print(f'  Next epoch           : {START_EPOCH}')
    print(f'  Best val AUC so far  : {best_val_auc:.4f}')
    print(f'  Patience counter     : {patience_count}/{PATIENCE}')
    print(f'  History length       : {len(history["train_loss"])} epochs logged')
    print(f'  Remaining epochs     : {NUM_EPOCHS - ckpt["epoch"]}')
    print('=' * 55)

    if START_EPOCH > NUM_EPOCHS:
        print(f'Training already complete ({NUM_EPOCHS} epochs done). '
              'Load best model in Cell 14.')
    elif patience_count >= PATIENCE:
        print('Early stopping condition was already met. '
              'Load best model in Cell 14.')
    else:
        print(f'Resuming from epoch {START_EPOCH}...')
else:
    print('No checkpoint found — starting fresh from epoch 1.')

In [13]:
# Cell 11: Train/Eval Functions
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    all_preds, all_targets = [], []
    pbar = tqdm(loader, desc='Training', leave=False, mininterval=2.0)  # ⚡ Reduced tqdm overhead
    for left, right, meta, labels in pbar:
        left   = left.to(DEVICE, non_blocking=True)
        right  = right.to(DEVICE, non_blocking=True)
        meta   = meta.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler:
            with torch.cuda.amp.autocast(dtype=torch.float16):  # ⚡ Explicit FP16
                logits = model(left, right, meta)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(left, right, meta)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_targets.append(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    p = np.vstack(all_preds); t = np.vstack(all_targets)
    try:    a = roc_auc_score(t, p, average='macro')
    except: a = 0.0
    f = f1_score(t, (p>0.5).astype(int), average='macro', zero_division=0)
    return total_loss / len(loader), a, f

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    for left, right, meta, labels in loader:
        left   = left.to(DEVICE, non_blocking=True)
        right  = right.to(DEVICE, non_blocking=True)
        meta   = meta.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(dtype=torch.float16):  # ⚡ FP16 inference
            logits = model(left, right, meta)
        total_loss += criterion(logits, labels).item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(labels.cpu().numpy())
    p = np.vstack(all_preds); t = np.vstack(all_targets)
    try:    a = roc_auc_score(t, p, average='macro')
    except: a = 0.0
    f = f1_score(t, (p>0.5).astype(int), average='macro', zero_division=0)
    return total_loss / len(loader), a, f, p, t

print('Train/eval functions ready.')

In [15]:
# ============================================================
# Cell 12: TRAINING LOOP with checkpoint save every epoch
# ============================================================

def save_checkpoint(path, epoch, model, optimizer, scheduler, scaler,
                    best_val_auc, patience_count, history, val_auc, val_f1):
    """
    Saves complete training state.
    Uses atomic write (tmp -> rename) to prevent corrupt files
    if the save itself is interrupted.
    """
    state = {
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_auc':         best_val_auc,
        'patience_count':       patience_count,
        'history':              history,
        'val_auc':              val_auc,
        'val_f1':               val_f1,
    }
    if scaler is not None:
        state['scaler_state_dict'] = scaler.state_dict()

    # Atomic write: save to .tmp first, then rename
    # If save is interrupted mid-write, the .tmp stays incomplete
    # and the previous good checkpoint is untouched
    tmp = path + '.tmp'
    torch.save(state, tmp)
    os.replace(tmp, path)  # atomic on Linux


# Guard: skip if training already done
if START_EPOCH > NUM_EPOCHS or patience_count >= PATIENCE:
    print('Training already complete or early stopped. Skip to Cell 13.')
else:
    print('=' * 65)
    print(f'{"Epoch":>6} | {"Tr Loss":>8} | {"Tr AUC":>7} | '
          f'{"Val Loss":>8} | {"Val AUC":>7} | {"Val F1":>7} | Status')
    print('=' * 65)

    for epoch in range(START_EPOCH, NUM_EPOCHS + 1):

        # Train
        tr_loss, tr_auc, tr_f1 = train_epoch(
            model, train_loader, optimizer, criterion, scaler)

        # Validate
        val_loss, val_auc, val_f1, _, _ = evaluate(
            model, val_loader, criterion)

        # Step LR scheduler
        scheduler.step(epoch)

        # Append to history
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_auc'].append(tr_auc)
        history['val_auc'].append(val_auc)
        history['train_f1'].append(tr_f1)
        history['val_f1'].append(val_f1)

        # Check if this is the best epoch
        is_best = val_auc > best_val_auc
        if is_best:
            best_val_auc   = val_auc
            patience_count = 0
            status = 'BEST'
        else:
            patience_count += 1
            status = f'patience {patience_count}/{PATIENCE}'

        print(f'{epoch:>6} | {tr_loss:>8.4f} | {tr_auc:>7.4f} | '
              f'{val_loss:>8.4f} | {val_auc:>7.4f} | {val_f1:>7.4f} | {status}')

        # ---- SAVE LATEST CHECKPOINT (every single epoch) ----
        # This is what enables resume — always reflects the most
        # recently COMPLETED epoch
        save_checkpoint(
            LATEST_CKPT, epoch, model, optimizer, scheduler, scaler,
            best_val_auc, patience_count, history, val_auc, val_f1
        )

        # ---- SAVE BEST CHECKPOINT (only when AUC improves) ----
        # This is what Cell 14 loads for final evaluation
        if is_best:
            save_checkpoint(
                BEST_CKPT, epoch, model, optimizer, scheduler, scaler,
                best_val_auc, patience_count, history, val_auc, val_f1
            )
            print(f'         → best_model.pth updated (AUC {best_val_auc:.4f})')

        # ---- Early stopping ----
        if patience_count >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}. Best Val AUC: {best_val_auc:.4f}')
            break

    print('=' * 65)
    print(f'Training complete. Best Val AUC : {best_val_auc:.4f}')
    print(f'Best model saved : {BEST_CKPT}')
    print(f'Latest saved     : {LATEST_CKPT}')

In [ ]:
# Cell 13: Training Curves
epochs_ran = len(history['train_loss'])
x          = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('BilateralFusionNet Training History', fontsize=14, fontweight='bold')

axes[0].plot(x, history['train_loss'], 'b-o', ms=3, label='Train')
axes[0].plot(x, history['val_loss'],   'r-o', ms=3, label='Val')
axes[0].set_title('Focal Loss'); axes[0].legend(); axes[0].grid(0.3)

axes[1].plot(x, history['train_auc'], 'b-o', ms=3, label='Train')
axes[1].plot(x, history['val_auc'],   'r-o', ms=3, label='Val')
axes[1].axhline(max(history['val_auc']), color='green', linestyle='--',
                label=f'Best: {max(history["val_auc"]):.4f}')
axes[1].set_title('Macro AUC-ROC'); axes[1].legend(); axes[1].grid(0.3)

axes[2].plot(x, history['train_f1'], 'b-o', ms=3, label='Train')
axes[2].plot(x, history['val_f1'],   'r-o', ms=3, label='Val')
axes[2].set_title('Macro F1-Score'); axes[2].legend(); axes[2].grid(0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# Cell 14: Load Best Model and Evaluate on Test Set
print(f'Loading best model from: {BEST_CKPT}')
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'  From epoch : {ckpt["epoch"]}')
print(f'  Val AUC    : {ckpt["val_auc"]:.4f}')
print(f'  Val F1     : {ckpt["val_f1"]:.4f}')

test_loss, test_auc, test_f1, test_preds, test_targets = evaluate(
    model, test_loader, criterion)
test_bin_preds = (test_preds > 0.5).astype(int)

print('\n--- TEST SET RESULTS ---')
print(f'Test Loss      : {test_loss:.4f}')
print(f'Macro AUC-ROC  : {test_auc:.4f}')
print(f'Macro F1-Score : {test_f1:.4f}')

print('\nPer-class AUC:')
for i, (abbr, name) in enumerate(zip(CLASS_ABBR, CLASSES)):
    try:
        ca = roc_auc_score(test_targets[:,i], test_preds[:,i])
        print(f'  {abbr} {name[:30]:30s}: {ca:.4f}')
    except:
        print(f'  {abbr} {name[:30]:30s}: N/A')

print('\n', classification_report(test_targets, test_bin_preds,
                                   target_names=CLASS_ABBR, zero_division=0))

In [ ]:
# Cell 15: ROC Curves + Confusion Matrices
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Per-Class ROC Curves', fontsize=14, fontweight='bold')
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i, (abbr, name, ax, color) in enumerate(zip(CLASS_ABBR, CLASSES, axes.flat, colors)):
    try:
        fpr, tpr, _ = roc_curve(test_targets[:,i], test_preds[:,i])
        ra = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'AUC={ra:.3f}')
        ax.fill_between(fpr, tpr, alpha=0.1, color=color)
    except: ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
    ax.plot([0,1],[0,1],'k--', alpha=0.3); ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
    ax.set_title(f'{abbr}: {name[:20]}', fontsize=9, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(0.2)
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

from sklearn.metrics import multilabel_confusion_matrix
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Per-Class Confusion Matrices', fontsize=14, fontweight='bold')
mcm = multilabel_confusion_matrix(test_targets, test_bin_preds)
for i, (abbr, ax) in enumerate(zip(CLASS_ABBR, axes.flat)):
    cm = mcm[i]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])
    tn,fp,fn,tp = cm.ravel()
    ax.set_title(f'{abbr} | Sens:{tp/(tp+fn+1e-6):.2f} Spec:{tn/(tn+fp+1e-6):.2f}', fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 16: Grad-CAM
class LeftEyeWrapper(nn.Module):
    def __init__(self, model, fixed_right, fixed_meta):
        super().__init__()
        self.model = model; self.fr = fixed_right; self.fm = fixed_meta
    def forward(self, left): return self.model(left, self.fr, self.fm)

class RightEyeWrapper(nn.Module):
    def __init__(self, model, fixed_left, fixed_meta):
        super().__init__()
        self.model = model; self.fl = fixed_left; self.fm = fixed_meta
    def forward(self, right): return self.model(self.fl, right, self.fm)

inv_norm = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225])

def generate_gradcam(model, dataset, target_class_idx=1, num_samples=4):
    model.eval()
    abbr = CLASS_ABBR[target_class_idx]
    pos  = [i for i in range(len(dataset)) if dataset.df.iloc[i][abbr]==1]
    if not pos: print(f'No positive samples for {abbr}'); return
    sel  = pos[:min(num_samples, len(pos))]
    tgt_layer = model.encoder.blocks[-1]
    fig, axes = plt.subplots(len(sel), 4, figsize=(20, 4*len(sel)))
    if len(sel)==1: axes = axes.reshape(1,-1)
    fig.suptitle(f'Grad-CAM | Class: {CLASSES[target_class_idx]}\n'
                 'Left | Left-CAM | Right | Right-CAM', fontsize=12, fontweight='bold')
    for ri, si in enumerate(sel):
        l, r, m, lb = dataset[si]
        lt = l.unsqueeze(0).to(DEVICE); rt = r.unsqueeze(0).to(DEVICE); mt = m.unsqueeze(0).to(DEVICE)
        tgt = [ClassifierOutputTarget(target_class_idx)]
        lw  = LeftEyeWrapper(model, rt, mt)
        with GradCAM(model=lw, target_layers=[tgt_layer]) as cam:
            cl = cam(lt, tgt)[0]
        rw  = RightEyeWrapper(model, lt, mt)
        with GradCAM(model=rw, target_layers=[tgt_layer]) as cam:
            cr = cam(rt, tgt)[0]
        lr = np.clip(inv_norm(l).permute(1,2,0).numpy(), 0, 1)
        rr = np.clip(inv_norm(r).permute(1,2,0).numpy(), 0, 1)
        al = [CLASS_ABBR[i] for i in range(NUM_CLASSES) if lb[i]==1]
        for ci, (img, title) in enumerate([
            (lr,  'Left Eye'),
            (show_cam_on_image(lr, cl, use_rgb=True), f'Left CAM→{abbr}'),
            (rr,  'Right Eye'),
            (show_cam_on_image(rr, cr, use_rgb=True), f'Right CAM→{abbr}')
        ]):
            axes[ri,ci].imshow(img); axes[ri,ci].axis('off')
            axes[ri,ci].set_title(title, fontsize=9)
            if ci==0: axes[ri,ci].set_ylabel(f'Labels:{al}', fontsize=8)
    plt.tight_layout()
    p = f'/kaggle/working/gradcam_{abbr}.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved {p}')

generate_gradcam(model, test_dataset, target_class_idx=1, num_samples=4)  # DR
generate_gradcam(model, test_dataset, target_class_idx=2, num_samples=4)  # Glaucoma

In [ ]:
# Cell 17: SHAP Metadata Analysis
class MetadataForSHAP(nn.Module):
    def __init__(self, model, bilateral_mean):
        super().__init__()
        self.model    = model
        self.bil_mean = bilateral_mean
    def forward(self, meta):
        B   = meta.shape[0]
        bil = self.bil_mean.expand(B, -1)
        fm  = self.model.meta_mlp(meta)
        return torch.sigmoid(self.model.classifier(torch.cat([bil, fm], dim=1)))

model.eval()
meta_list, bil_list = [], []
with torch.no_grad():
    for i in range(min(80, len(test_dataset))):
        l, r, m, _ = test_dataset[i]
        lt = l.unsqueeze(0).to(DEVICE); rt = r.unsqueeze(0).to(DEVICE)
        fl = model.encoder(lt); fr = model.encoder(rt)
        fb = model.cross_attn(fl, fr)
        meta_list.append(m.numpy())
        bil_list.append(fb.cpu())

meta_arr  = np.array(meta_list)
bil_mean  = torch.stack(bil_list).mean(0).to(DEVICE)
shap_mdl  = MetadataForSHAP(model, bil_mean).to(DEVICE)
shap_mdl.eval()

def model_fn(x):
    with torch.no_grad():
        return shap_mdl(torch.tensor(x, dtype=torch.float32).to(DEVICE)).cpu().numpy()

print('Computing SHAP values...')
explainer  = shap.KernelExplainer(model_fn, meta_arr[:30])
shap_vals  = explainer.shap_values(meta_arr[30:50], nsamples=100)

feat_names = ['Age (norm)', 'Gender (M=1)']
fig, axes  = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('SHAP — Metadata Feature Importance per Disease Class\n'
             '(Novel: shows how age & gender influence each prediction)',
             fontsize=12, fontweight='bold')
for i, (abbr, name, ax) in enumerate(zip(CLASS_ABBR, CLASSES, axes.flat)):
    sv   = np.array(shap_vals[i])
    mabs = np.abs(sv).mean(axis=0)
    bars = ax.barh(feat_names, mabs, color=['steelblue','coral'])
    mv   = sv.mean(axis=0)
    for bar, val, ma in zip(bars, mv, mabs):
        ax.text(ma+0.001, bar.get_y()+bar.get_height()/2,
                f'{"↑" if val>0 else "↓"}{abs(val):.3f}', va='center', fontsize=8)
    ax.set_title(f'{abbr}: {name[:20]}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Mean |SHAP|'); ax.grid(0.2, axis='x')
plt.tight_layout()
plt.savefig('/kaggle/working/shap_metadata.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSHAP summary:')
for i, (abbr, name) in enumerate(zip(CLASS_ABBR, CLASSES)):
    sv = np.array(shap_vals[i])
    print(f'  {abbr}: Age={sv[:,0].mean():+.4f}  Gender={sv[:,1].mean():+.4f}')

In [ ]:
# Cell 18: Ablation Study
class AblationModel(nn.Module):
    def __init__(self, mode='full', nc=8, dropout=0.3):
        super().__init__()
        self.mode    = mode
        self.encoder = timm.create_model('efficientnet_b3', pretrained=True,
                                         num_classes=0, global_pool='avg')
        fd = self.encoder.num_features
        nh = 8 if fd%8==0 else 4
        if mode == 'full':
            self.cross_attn = CrossAttentionFusion(fd, nh, 0.1)
            self.meta_mlp   = MetadataMLP(2, 64, 128, 0.2)
            fdim = fd*2+128
        elif mode == 'no_meta':
            self.cross_attn = CrossAttentionFusion(fd, nh, 0.1)
            fdim = fd*2
        elif mode == 'no_attn':
            self.meta_mlp = MetadataMLP(2, 64, 128, 0.2)
            fdim = fd*2+128
        elif mode == 'baseline':
            fdim = fd
        self.classifier = nn.Sequential(
            nn.Linear(fdim,512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(dropout/2),
            nn.Linear(256,nc))

    def forward(self, l, r, m):
        fl = self.encoder(l); fr = self.encoder(r)
        if self.mode=='full':
            fb = self.cross_attn(fl, fr); fm = self.meta_mlp(m)
            return self.classifier(torch.cat([fb,fm],1))
        elif self.mode=='no_meta':
            return self.classifier(self.cross_attn(fl,fr))
        elif self.mode=='no_attn':
            return self.classifier(torch.cat([fl,fr,self.meta_mlp(m)],1))
        elif self.mode=='baseline':
            return self.classifier(fl)

def quick_eval(mode, epochs=5):
    m   = AblationModel(mode, NUM_CLASSES).to(DEVICE)
    opt = optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = MultiLabelFocalLoss(0.25, 2.0, pos_weight_tensor)
    for _ in range(epochs): train_epoch(m, train_loader, opt, crit, scaler)
    _, a, f, _, _ = evaluate(m, test_loader, crit)
    return a, f

print('Running ablation (5 epochs each, ~10-15 min)...')
abl = {}
for name, mode in [
    ('Baseline (left eye only)',      'baseline'),
    ('+ Bilateral (no attention)',    'no_attn'),
    ('+ Cross-attention (no meta)',   'no_meta'),
    ('Full BilateralFusionNet',       'full'),
]:
    print(f'  {name}...', end=' ', flush=True)
    a, f = quick_eval(mode)
    abl[name] = {'AUC': a, 'F1': f}
    print(f'AUC={a:.4f} F1={f:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Ablation Study — Contribution of Each Component', fontsize=13, fontweight='bold')
names  = list(abl.keys())
clrs   = ['#7570b3','#7570b3','#7570b3','#d95f02']
for i, (metric, axi) in enumerate([('AUC','AUC-ROC'),('F1','F1-Score')]):
    vals = [abl[n]['AUC' if metric=='AUC' else 'F1'] for n in names]
    bars = ax[i].barh(names, vals, color=clrs)
    ax[i].set_title(metric); ax[i].set_xlim(0.4, 1.0)
    for b, v in zip(bars, vals):
        ax[i].text(v+0.005, b.get_y()+b.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
    ax[i].grid(0.2, axis='x')
plt.tight_layout()
plt.savefig('/kaggle/working/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 19: Comparison Table
comp = [
    ('DEEPOcuDx DenseNet121',     'ODIR-5K',  0.61,   0.56,   None,       'ICDSINC 2025'),
    ('DEEPOcuDx Swin Transformer','ODIR-5K',  0.61,   0.54,   None,       'ICDSINC 2025'),
    ('InvoFundusNet',             'Eye-Dis.', 0.9947, None,   0.9521,     'ASET 2025'),
    ('EyeLineNet Ens+CBAM',       'Eye-Dis.', 0.935,  None,   None,       'ICIIP 2025'),
    ('Hybrid CNN+ViT Ensemble',   'Eye-Dis.', 0.9912, None,   None,       'AIBThings 2025'),
    ('BilateralFusionNet (Ours)', 'ODIR-5K',  None,   round(test_f1,4), round(test_auc,4), 'This work'),
]
cdf = pd.DataFrame(comp, columns=['Method','Dataset','Accuracy','Macro F1','Macro AUC','Source'])
print(cdf.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')
tbl = ax.table(cellText=cdf.values, colLabels=cdf.columns, cellLoc='center', loc='center',
               bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for j in range(len(cdf.columns)):
    tbl[(len(cdf),j)].set_facecolor('#d4edda')
    tbl[(len(cdf),j)].set_text_props(fontweight='bold')
plt.title('Comparison with Prior Works (green = ours)', fontsize=12, fontweight='bold', pad=15)
plt.savefig('/kaggle/working/comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 20: Final Summary
print('=' * 60)
print('BILATERALFUSIONNET — COMPLETE')
print('=' * 60)
print(f'Test Macro AUC-ROC : {test_auc:.4f}')
print(f'Test Macro F1      : {test_f1:.4f}')
print(f'Test Loss          : {test_loss:.4f}')
print()
print('Output files:')
for f in [
    BEST_CKPT, LATEST_CKPT,
    '/kaggle/working/eda_analysis.png',
    '/kaggle/working/training_curves.png',
    '/kaggle/working/roc_curves.png',
    '/kaggle/working/confusion_matrices.png',
    '/kaggle/working/gradcam_D.png',
    '/kaggle/working/gradcam_G.png',
    '/kaggle/working/shap_metadata.png',
    '/kaggle/working/ablation_study.png',
    '/kaggle/working/comparison_table.png',
]:
    print(f'  {"✓" if Path(f).exists() else "✗"} {f}')

pred_df = test_df.copy().reset_index(drop=True)
for i, a in enumerate(CLASS_ABBR):
    pred_df[f'pred_{a}']     = test_preds[:,i]
    pred_df[f'pred_bin_{a}'] = test_bin_preds[:,i]
pred_df.to_csv('/kaggle/working/test_predictions.csv', index=False)
print('  ✓ /kaggle/working/test_predictions.csv')